In [1]:
import pandas as pd
import pickle
import re
from tqdm import tqdm

# โหลด Cache เดิมขึ้นมา
CACHE_FILE = 'resource/cleaned_recipes.pkl'
with open(CACHE_FILE, 'rb') as f:
    df = pickle.load(f)

# ฟังก์ชันดึง URL ออกมาจากรูปแบบของภาษา R
def extract_first_image(img_str):
    if not isinstance(img_str, str) or img_str == 'character(0)' or img_str.strip() == '':
        return None

    # ใช้ Regex ดึง URL ที่อยู่ในเครื่องหมายคำพูด "..."
    urls = re.findall(r'"(https?://.*?)"', img_str)
    if urls:
        return urls[0] # เอาแค่ URL แรกสุดรูปเดียวพอ
    return None

# สร้างคอลัมน์ใหม่ที่เก็บแค่ URL สะอาดๆ
print("Cleaning image strings...")
df['Images_clean'] = df['Images'].apply(extract_first_image)

# นับจำนวนที่มีและไม่มีรูป
has_img_count = df['Images_clean'].notna().sum()
no_img_count = df['Images_clean'].isna().sum()

print(f"มีรูปภาพพร้อมใช้: {has_img_count:,} เมนู")
print(f"ไม่มีรูปภาพ: {no_img_count:,} เมนู")

Cleaning image strings...
✅ มีรูปภาพพร้อมใช้: 165,896 เมนู
❌ ไม่มีรูปภาพ: 356,621 เมนู


In [16]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
from Search_PJ import get_and_clean_data

In [13]:
BATCH_SIZE   = 5_000   # rows processed per chunk (tune to your RAM)
TOP_K        = 1       # how many nearest neighbors to retrieve
N_JOBS       = -1      # use all CPU cores
OUTPUT_FILE  = Path('resource/borrowed_images.parquet')

In [14]:
def make_query_text(df: pd.DataFrame) -> pd.Series:
    name  = df['Name_clean'].fillna('')
    ingr  = df['RecipeIngredientParts_clean'].fillna('')
    kw    = df['Keywords_clean'].fillna('')
    # Weight: name × 4, ingredients × 2, keywords × 1
    return (name + ' ') * 4 + (ingr + ' ') * 2 + kw

In [21]:
def extract_first_image(img_str):
    if not isinstance(img_str, str) or img_str == 'character(0)' or img_str.strip() == '':
        return None
    urls = re.findall(r'"(https?://.*?)"', img_str)
    return urls[0] if urls else None


def build_image_borrowing_map(data: pd.DataFrame) -> pd.DataFrame:
    # ── 1. Clean images & Split ───────────────────────────────────────────────
    data = data.copy()
    data['Images_clean'] = data['Images'].apply(extract_first_image)

    has_img = data[data['Images_clean'].notna()].copy()
    no_img  = data[data['Images_clean'].isna()].copy()
    print(f"Image pool : {len(has_img):,}")
    print(f"No-image   : {len(no_img):,}")

    # ── 2. Build combined text ────────────────────────────────────────────────
    pool_text  = make_query_text(has_img)
    noimg_text = make_query_text(no_img)

    # ── 3. Fit TF-IDF on the image pool ONLY ─────────────────────────────────
    print("Fitting TF-IDF on image pool...")
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=3,
        max_features=80_000,
        sublinear_tf=True,
    )
    pool_matrix = vectorizer.fit_transform(pool_text)
    print(f"  Matrix shape: {pool_matrix.shape}, nnz={pool_matrix.nnz:,}")

    # ── 4. Fit NearestNeighbors ───────────────────────────────────────────────
    print("Fitting NearestNeighbors...")
    nn = NearestNeighbors(
        n_neighbors=TOP_K,
        metric='cosine',
        algorithm='brute',
        n_jobs=N_JOBS,
    )
    nn.fit(pool_matrix)

    pool_ids    = has_img['RecipeId'].values
    pool_images = has_img['Images_clean'].values  # already clean URLs now

    # ── 5. Process in batches ─────────────────────────────────────────────────
    records   = []
    noimg_ids = no_img['RecipeId'].values

    for start in tqdm(range(0, len(no_img), BATCH_SIZE), desc='Matching batches'):
        end          = min(start + BATCH_SIZE, len(no_img))
        batch_matrix = vectorizer.transform(noimg_text.iloc[start:end])
        distances, indices = nn.kneighbors(batch_matrix)

        for i, (dist_row, idx_row) in enumerate(zip(distances, indices)):
            records.append({
                'RecipeId'     : int(noimg_ids[start + i]),
                'BorrowedFrom' : int(pool_ids[idx_row[0]]),
                'BorrowedImage': str(pool_images[idx_row[0]]),  # clean URL directly
                'similarity'   : float(1.0 - dist_row[0]),
            })

    result_df = pd.DataFrame(records)
    result_df.to_parquet(OUTPUT_FILE, index=False)
    print(f"\nSaved → {OUTPUT_FILE}")
    print(result_df.describe())
    return result_df

In [22]:
if __name__ == '__main__':
    data = get_and_clean_data()
    mapping = build_image_borrowing_map(data)

Loading cleaned data from cache...
Loaded 522,517 recipes from cache
Image pool : 165,896
No-image   : 356,621
Fitting TF-IDF on image pool...
  Matrix shape: (165896, 9177), nnz=3,607,524
Fitting NearestNeighbors...


Matching batches: 100%|██████████| 72/72 [19:40<00:00, 16.39s/it]



Saved → resource\borrowed_images.parquet
            RecipeId   BorrowedFrom     similarity
count  356621.000000  356621.000000  356621.000000
mean   273862.915008  247395.038352       0.607890
std    155475.212852  154640.754645       0.111065
min        43.000000      39.000000       0.267932
25%    139124.000000  112559.000000       0.527294
50%    277866.000000  235316.000000       0.594629
75%    408731.000000  372288.000000       0.675835
max    541383.000000  541380.000000       1.000000


In [3]:
import pandas as pd
import re

def extract_first_image(val):
    if pd.isna(val) or val == 'character(0)' or str(val).strip() == '':
        return None
    # Extract only the URL within quotes
    urls = re.findall(r'"(https?://.*?)"', str(val))
    return urls[0] if urls else None

def parse_r_list(val):
    if pd.isna(val) or val == 'character(0)' or str(val).strip() == '':
        return ""
    # Extract all text within quotes and join them with spaces
    items = re.findall(r'"([^"]+)"', str(val))
    return " ".join(items) if items else str(val)

def update_csv_with_borrowed_images():
    csv_path = 'resource/recipes.csv'
    parquet_path = 'resource/borrowed_images.parquet'
    output_path = 'resource/recipes_updated.csv'

    print("1. Loading borrowed_images.parquet...")
    try:
        borrowed_df = pd.read_parquet(parquet_path)
    except FileNotFoundError:
        print("Parquet file not found!")
        return

    # No need to wrap with c("..."), just use the raw URL
    borrow_map = borrowed_df.set_index('RecipeId')['BorrowedImage'].to_dict()

    print("2. Reading and writing in chunks while converting R format...")
    first_chunk = True
    total_filled = 0

    for chunk in pd.read_csv(csv_path, chunksize=10000):

        # --- Clean up R format in important columns ---
        chunk['Images'] = chunk['Images'].apply(extract_first_image)
        chunk['RecipeIngredientParts'] = chunk['RecipeIngredientParts'].apply(parse_r_list)
        chunk['RecipeInstructions'] = chunk['RecipeInstructions'].apply(parse_r_list)

        # --- Handle missing images ---
        is_missing = chunk['Images'].isna()
        total_filled += is_missing.sum()

        # Fill in the borrowed images
        chunk.loc[is_missing, 'Images'] = chunk.loc[is_missing, 'RecipeId'].map(borrow_map)

        # Write to the new file
        chunk.to_csv(
            output_path,
            index=False,
            mode='w' if first_chunk else 'a',
            header=first_chunk
        )
        first_chunk = False

    print(f"Successfully filled {total_filled:,} images")
    print(f"Saved and R format removed successfully at {output_path}")

if __name__ == '__main__':
    update_csv_with_borrowed_images()

1. โหลด borrowed_images.parquet...
2. อ่านและเขียนแบบ chunk พร้อมแปลงฟอร์แมตภาษา R...
✅ เติมรูปสำเร็จ 356,621 รูป
💾 บันทึกและกำจัดฟอร์แมต R เรียบร้อย ไว้ที่ resource/recipes_updated.csv
